In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from math import sqrt

df = pd.read_csv('elite_supercars_cleaned.csv')

target_var = 'Price'
X = df.drop(columns=[target_var])
y = df[target_var]

log_transformed_target = y.min() < 1 or y.max() < 20
if log_transformed_target:
    print("⚠️ Detected log-transformed target (Price). Model will exponentiate predictions.")

numeric_cols = [c for c in X.columns if np.issubdtype(X[c].dtype, np.number)]
categorical_cols = [c for c in X.columns if X[c].dtype == 'object' or X[c].dtype == 'bool']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


numeric_pipeline = Pipeline([('scaler', StandardScaler())])
categorical_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ]
)


models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42)
}


cv = KFold(n_splits=5, shuffle=True, random_state=42)


results = []
for name, model in models.items():
    pipeline = Pipeline([('preprocess', preprocessor), ('model', model)])
    r2 = cross_val_score(pipeline, X_train, y_train, scoring='r2', cv=cv, n_jobs=-1).mean()
    mae = -cross_val_score(pipeline, X_train, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1).mean()
    mse = -cross_val_score(pipeline, X_train, y_train, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1).mean()
    rmse = sqrt(mse)
    results.append({'Model': name, 'CV R2': r2, 'CV MAE': mae, 'CV RMSE': rmse})

results_df = pd.DataFrame(results)
print("Model CV Comparison:\n", results_df)

best_model_name = results_df.sort_values('CV R2', ascending=False).iloc[0]['Model']
best_pipeline = Pipeline([('preprocess', preprocessor), ('model', models[best_model_name])])

best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
test_rmse = sqrt(test_mse)

print(f"\nBest Model: {best_model_name}")
print(f"Test R2: {test_r2:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

joblib.dump(best_pipeline, f'best_model_{best_model_name}.pkl')

new_sample = {
    'Brand': 'Ferrari',
    'Model': '488 GTB',
    'Year': 2021,
    'Engine_Size': 3.9,
    'Horsepower': 661,
    'Torque': 760,
    'Top_Speed': 330,
    'Acceleration_0_100': 3.0,
    'Transmission': 'Automatic',
    'Drivetrain': 'RWD',
    'Fuel_Type': 'Petrol',
    'Fuel_Efficiency': 8.8,
    'CO2_Emissions': 307,
    'Mileage': 5000,
    'Safety_Rating': 5,
    'Insurance_Cost': 12000,
    'Country': 'Italy',
    'Number_of_Owners': 1,
    'Condition': 'new',
    'Popularity': 'High',
    'Production_Units': 2000,
    'Market_Demand': 'High',
    'Weight': 1475
}

new_df = pd.DataFrame([new_sample])

for col in X.columns:
    if col not in new_df.columns:
        if X[col].dtype == 'object' or X[col].dtype == 'bool':
            new_df[col] = ''
        else:
            new_df[col] = 0

new_df = new_df[X.columns]

predicted_value = best_pipeline.predict(new_df)

if log_transformed_target:
    predicted_value = np.exp(predicted_value)

predicted_value = np.maximum(predicted_value, 0)

print("\nPredicted Market Price (Actual):", predicted_value[0])


⚠️ Detected log-transformed target (Price). Model will exponentiate predictions.
Model CV Comparison:
               Model     CV R2    CV MAE   CV RMSE
0  LinearRegression  0.877546  0.293652  0.349898
1             Ridge  0.877551  0.293685  0.349890
2             Lasso  0.879548  0.293175  0.347016
3        ElasticNet  0.879543  0.293082  0.347021
4      RandomForest  0.999999  0.000865  0.001126

Best Model: RandomForest
Test R2: 1.0000
Test MAE: 0.0007
Test RMSE: 0.0009

Predicted Market Price (Actual): 0.17287942742256857


In [ ]:
import pandas as pd
import numpy as np

df_original = pd.read_csv('Elite Sports Cars in Data.csv')

price_mean = df_original['Price'].mean()
price_std = df_original['Price'].std()

print("Original Price Mean:", price_mean)
print("Original Price Std:", price_std)

predicted_value = predicted_value[0]

currency_price = (predicted_value * price_std) + price_mean

print(f"Predicted Market Price (Currency): ₹{currency_price:,.2f}")


Original Price Mean: 262067.3294
Original Price Std: 137678.80390601992
Predicted Market Price (Currency): ₹285,869.88
